# Clinical Pharmacy Assistant OS — Google Colab
نسخة عربية RTL بواجهة ويب مؤقتة عبر Gradio، وقاعدة SQLite، واختبارات أساسية.

**تنبيه:** نموذج أولي تعليمي غير معتمد لاتخاذ قرارات علاجية. قواعد التداخلات محدودة وغير متحققة استقلاليًا.

## طريقة التشغيل
# تعليمات التشغيل من هاتف POCO

1. افتح https://colab.research.google.com/
2. اختر **New notebook**.
3. من قائمة File اختر **Upload notebook** وارفع الملف `Clinical_Pharmacy_Assistant_OS_Colab.ipynb`.
4. شغّل الخلية الأولى لتثبيت Gradio.
5. (اختياري) إذا أردت الاحتفاظ بقاعدة البيانات في Google Drive، شغّل خلية Drive الاختيارية قبل خلية التطبيق، ثم عدّل `DB_PATH` في خلية التطبيق إلى:
   `/content/drive/MyDrive/ClinicalPharmacyOS/clinical_pharmacy.db`
6. شغّل خلية التطبيق الكبيرة. وافق على تثبيت الحزم إن طلب Colab ذلك.
7. انتظر حتى يظهر رابط عام مؤقت من Gradio (`https://....gradio.live`) وافتحه. الرابط مؤقت ويعمل ما دامت جلسة Colab نشطة.
8. من تبويب **اختبارات النظام** اضغط **تشغيل الاختبارات**. المتوقع 8 اختبارات ناجحة إذا اكتمل التشغيل دون أخطاء.
9. جرّب إنشاء ملف مريض برمز غير معرّف، ثم أضف الأدوية من تبويب إضافة دواء، وبعدها افحص التداخلات.
10. استخدم تبويب النسخ الاحتياطي لتنزيل ملف SQLite.

## ملاحظات مهمة
- هذا دفتر جاهز بالكود، لكن لم يتم تشغيله داخل حساب Google الخاص بك. قد تتغير واجهة Colab أو توافق الحزم بمرور الوقت.
- الرابط العام المؤقت لا ينبغي استخدامه مع بيانات مرضى حقيقية أو حساسة.
- قواعد التداخلات الثلاث تجريبية ومعلّمة بوضوح أنها غير متحققة استقلاليًا؛ ليست قاعدة تداخلات شاملة.
- الحاسبة تنفذ عملية حسابية فقط ولا تحدد جرعة علاجية مناسبة.
- سجل المراجع هو سجل إداري؛ لا يثبت أن المصدر قد تم قراءته أو التحقق منه.
- لا يحتوي الإصدار على اتصال بـ Gemini أو Lexicomp أو Micromedex أو أي قاعدة تجارية.
- لا تشارك رابط Gradio العام؛ يمكن لمن يحصل عليه الوصول إلى الواجهة خلال فترة عمل الجلسة.


In [ ]:
%pip -q install "gradio>=5.20,<6"

## تخزين دائم اختياري في Google Drive

إذا رغبت ببقاء قاعدة البيانات بعد انتهاء جلسة Colab، شغّل الخلية التالية قبل تشغيل التطبيق، ثم اتبع التعليق داخلها لتعديل `DB_PATH` في خلية التطبيق.
لا تستخدم بيانات مرضى حقيقية في هذه النسخة التجريبية.

In [ ]:
# اختياري: تخزين قاعدة البيانات في Google Drive حتى تبقى بعد انتهاء جلسة Colab.
# شغّل هذه الخلية قبل خلية التطبيق، ثم غيّر DB_PATH في خلية التطبيق إلى:
# DB_PATH = "/content/drive/MyDrive/ClinicalPharmacyOS/clinical_pharmacy.db"
#
# من الأفضل عدم استخدام بيانات مرضى حقيقية في هذا النموذج الأولي.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ClinicalPharmacyOS', exist_ok=True)
print("تم تجهيز مجلد التخزين. عدّل DB_PATH في خلية التطبيق قبل تشغيلها.")

## التطبيق

شغّل الخلية التالية. ستظهر واجهة التطبيق ورابط Gradio مؤقت. يجب أن تبقى جلسة Colab نشطة.

In [ ]:
# Clinical Pharmacy Assistant OS — Google Colab edition
# Educational prototype only. Not a validated clinical decision support system.

import os, sqlite3, json, math, unittest, html
from datetime import datetime, timezone
import gradio as gr

# Optional persistence:
# For persistent storage, mount Google Drive in a separate cell and set DB_PATH
# to a folder inside your Drive. Do not store identifiable patient data in this prototype.
DB_PATH = "/content/clinical_pharmacy.db"

# These are illustrative starter rules only. They are not a complete interaction database.
# Before any clinical use, verify each rule against current, licensed references and
# have the rule reviewed by a qualified clinical pharmacist.
INTERACTION_RULES = [
    {
        "a": "carbamazepine", "b": "erythromycin",
        "severity": "مرتفع — يحتاج إلى مراجعة سريرية",
        "summary": "قد يثبط الإريثروميسين استقلاب الكاربامازيبين ويرفع مستواه، ما قد يزيد خطر السمية.",
        "action": "راجع الطبيب/الصيدلي السريري، وقيّم الأعراض ومستوى الكاربامازيبين عند الاستطباب. تحقّق من بديل مناسب ومن مرجع حديث. لا تغيّر العلاج آليًا.",
        "source": "قاعدة توضيحية غير موثقة داخل التطبيق؛ يلزم التحقق من مرجع دوائي حديث.",
        "verified": False
    },
    {
        "a": "erythromycin", "b": "warfarin",
        "severity": "مهم — يلزم تقييم ومراقبة",
        "summary": "قد يزيد الإريثروميسين تأثير الوارفارين وخطر النزف لدى بعض المرضى، وقد تتغير قيمة INR.",
        "action": "راجع الطبيب/الصيدلي المسؤول، وراقب INR وفق الخطة السريرية المعتمدة وحالة المريض. لا تعدّل جرعة الوارفارين آليًا.",
        "source": "قاعدة توضيحية غير موثقة داخل التطبيق؛ يلزم التحقق من مرجع دوائي حديث.",
        "verified": False
    },
    {
        "a": "carbamazepine", "b": "warfarin",
        "severity": "مهم — يلزم تقييم ومتابعة",
        "summary": "قد يحفّز الكاربامازيبين إنزيمات الاستقلاب ويقلل تأثير الوارفارين؛ وقد تتغير متطلبات الجرعة عند بدء أحد الدواءين أو إيقافه.",
        "action": "راقب INR حسب خطة الطبيب عند بدء العلاج أو تغييره أو إيقافه. لا تغيّر جرعة الوارفارين بناءً على هذه الرسالة وحدها.",
        "source": "قاعدة توضيحية غير موثقة داخل التطبيق؛ يلزم التحقق من مرجع دوائي حديث.",
        "verified": False
    }
]

def db():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys=ON")
    return conn

def init_db():
    with db() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS patients(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_code TEXT NOT NULL,
            age_years REAL,
            weight_kg REAL,
            sex TEXT DEFAULT 'غير محدد',
            allergies TEXT DEFAULT '',
            diagnoses TEXT DEFAULT '',
            notes TEXT DEFAULT '',
            created_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS medications(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER NOT NULL,
            generic_name TEXT NOT NULL,
            brand_name TEXT DEFAULT '',
            strength TEXT DEFAULT '',
            dose TEXT DEFAULT '',
            route TEXT DEFAULT '',
            frequency TEXT DEFAULT '',
            FOREIGN KEY(patient_id) REFERENCES patients(id) ON DELETE CASCADE
        );
        CREATE TABLE IF NOT EXISTS references_log(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            publisher TEXT DEFAULT '',
            edition TEXT DEFAULT '',
            url TEXT DEFAULT '',
            access_status TEXT DEFAULT 'غير محدد',
            verification_notes TEXT DEFAULT '',
            created_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS interaction_reviews(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER NOT NULL,
            result_json TEXT NOT NULL,
            created_at TEXT NOT NULL,
            FOREIGN KEY(patient_id) REFERENCES patients(id) ON DELETE CASCADE
        );
        """)
init_db()

def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def add_patient(patient_code, age_years, weight_kg, sex, allergies, diagnoses, notes):
    code = (patient_code or "").strip()
    if not code:
        return "أدخل رمزًا للمريض.", list_patients()
    try:
        age = float(age_years) if str(age_years or "").strip() else None
        weight = float(weight_kg) if str(weight_kg or "").strip() else None
        if age is not None and age < 0: raise ValueError
        if weight is not None and weight <= 0: raise ValueError
    except ValueError:
        return "تحقق من العمر والوزن: العمر غير سالب والوزن أكبر من صفر.", list_patients()
    with db() as c:
        cur = c.execute("""INSERT INTO patients
            (patient_code,age_years,weight_kg,sex,allergies,diagnoses,notes,created_at)
            VALUES(?,?,?,?,?,?,?,?)""",
            (code, age, weight, sex or "غير محدد", allergies or "", diagnoses or "",
             notes or "", now_utc()))
        pid = cur.lastrowid
    return f"تم إنشاء سجل تجريبي برقم {pid}. استخدم رمزًا غير معرّف ولا تدخل بيانات شخصية حقيقية.", list_patients()

def list_patients():
    with db() as c:
        rows = c.execute("""SELECT p.id,p.patient_code,p.age_years,p.weight_kg,p.sex,p.diagnoses,
            COUNT(m.id) AS medication_count
            FROM patients p LEFT JOIN medications m ON m.patient_id=p.id
            GROUP BY p.id ORDER BY p.id DESC""").fetchall()
    data = []
    for r in rows:
        data.append([r["id"], r["patient_code"], r["age_years"], r["weight_kg"], r["sex"], r["diagnoses"], r["medication_count"]])
    return data

def add_medication(patient_id, generic_name, brand_name, strength, dose, route, frequency):
    try:
        pid = int(patient_id)
    except (TypeError, ValueError):
        return "رقم ملف المريض مطلوب ويجب أن يكون رقمًا صحيحًا.", list_medications()
    generic = (generic_name or "").strip().lower()
    if not generic:
        return "الاسم العلمي مطلوب.", list_medications()
    with db() as c:
        if not c.execute("SELECT id FROM patients WHERE id=?", (pid,)).fetchone():
            return "لم يتم العثور على ملف بهذا الرقم.", list_medications()
        c.execute("""INSERT INTO medications
            (patient_id,generic_name,brand_name,strength,dose,route,frequency)
            VALUES(?,?,?,?,?,?,?)""",
            (pid, generic, brand_name or "", strength or "", dose or "", route or "", frequency or ""))
    return f"تمت إضافة الدواء إلى ملف رقم {pid}.", list_medications()

def list_medications():
    with db() as c:
        rows = c.execute("""SELECT m.id,m.patient_id,p.patient_code,m.generic_name,m.brand_name,
            m.strength,m.dose,m.route,m.frequency
            FROM medications m JOIN patients p ON p.id=m.patient_id ORDER BY m.id DESC""").fetchall()
    return [[r[k] for k in ("id","patient_id","patient_code","generic_name","brand_name","strength","dose","route","frequency")] for r in rows]

def check_interactions_text(medication_text):
    names = [x.strip().lower() for x in (medication_text or "").replace("،", ",").replace("\n", ",").split(",") if x.strip()]
    if len(names) < 2:
        return "أدخل اسمَي دواء على الأقل، وافصل بينها بفاصلة أو سطر جديد."
    found = []
    for rule in INTERACTION_RULES:
        if rule["a"] in names and rule["b"] in names:
            found.append(rule)
    lines = [
        "تنبيه: هذه مراجعة أولية لثلاث قواعد توضيحية فقط، وليست قاعدة تداخلات شاملة.",
        "عدم ظهور نتيجة لا يعني عدم وجود تداخل. لا تغيّر علاجًا اعتمادًا على هذا الفحص وحده.",
        ""
    ]
    if not found:
        lines.append("لم تطابق الأسماء أي قاعدة من القواعد المحدودة الموجودة في هذه النسخة.")
    for i, r in enumerate(found, 1):
        lines += [
            f"{i}. {r['a'].title()} + {r['b'].title()}",
            f"التصنيف الأولي: {r['severity']}",
            f"الشرح: {r['summary']}",
            f"إجراء مقترح للمراجعة: {r['action']}",
            f"المصدر/حالة التحقق: {r['source']}",
            "حالة القاعدة: غير متحققة استقلاليًا",
            ""
        ]
    return "\n".join(lines)

def check_patient_interactions(patient_id):
    try:
        pid = int(patient_id)
    except (TypeError, ValueError):
        return "أدخل رقم ملف مريض صحيحًا."
    with db() as c:
        patient = c.execute("SELECT * FROM patients WHERE id=?", (pid,)).fetchone()
        meds = c.execute("SELECT generic_name FROM medications WHERE patient_id=?", (pid,)).fetchall()
    if not patient:
        return "لم يتم العثور على ملف المريض."
    result = check_interactions_text(", ".join(m["generic_name"] for m in meds))
    with db() as c:
        c.execute("INSERT INTO interaction_reviews(patient_id,result_json,created_at) VALUES(?,?,?)",
                  (pid, json.dumps({"text": result}, ensure_ascii=False), now_utc()))
    return f"مراجعة الملف: {patient['patient_code']}\n\n{result}"

def dose_calculator(weight_kg, dose_mg_kg, max_dose_mg):
    try:
        weight = float(weight_kg)
        perkg = float(dose_mg_kg)
        maxdose = float(max_dose_mg) if str(max_dose_mg or "").strip() else None
        if not math.isfinite(weight) or not math.isfinite(perkg) or weight <= 0 or perkg < 0:
            raise ValueError
        if maxdose is not None and (not math.isfinite(maxdose) or maxdose < 0):
            raise ValueError
    except (ValueError, TypeError):
        return "مدخلات غير صالحة. الوزن أكبر من صفر والجرعة والحد الأقصى (إن وُجد) غير سالبين."
    raw = weight * perkg
    final = min(raw, maxdose) if maxdose is not None else raw
    return (
        f"الناتج الحسابي قبل تطبيق الحد الأقصى: {raw:.3f} mg لكل جرعة.\n"
        f"الناتج بعد تطبيق الحد الأقصى الذي أدخلته: {final:.3f} mg لكل جرعة.\n\n"
        "تحذير: هذه حاسبة ضرب حسابي وليست أداة لاختيار جرعة علاجية. يجب التحقق من المرجع والاستطباب "
        "والتكرار والجرعة القصوى ووظائف الأعضاء والعمر والشكل والتركيز الصيدلاني بصورة مستقلة."
    )

def add_reference(title, publisher, edition, url, access_status, verification_notes):
    if not (title or "").strip():
        return "اسم المرجع مطلوب.", list_references()
    with db() as c:
        c.execute("""INSERT INTO references_log
            (title,publisher,edition,url,access_status,verification_notes,created_at)
            VALUES(?,?,?,?,?,?,?)""",
            (title.strip(), publisher or "", edition or "", url or "",
             access_status or "غير محدد", verification_notes or "", now_utc()))
    return "تم تسجيل المرجع. التسجيل لا يعني التحقق من محتواه أو الترخيص.", list_references()

def list_references():
    with db() as c:
        rows = c.execute("SELECT id,title,publisher,edition,url,access_status,verification_notes,created_at FROM references_log ORDER BY id DESC").fetchall()
    return [[r[k] for k in ("id","title","publisher","edition","url","access_status","verification_notes","created_at")] for r in rows]

def run_tests():
    suite = unittest.TestSuite()
    class BasicTests(unittest.TestCase):
        def test_known_demo_pair(self):
            self.assertIn("Carbamazepine + Erythromycin", check_interactions_text("carbamazepine, erythromycin"))
        def test_unlisted_pair(self):
            self.assertIn("لم تطابق", check_interactions_text("aspirin, paracetamol"))
        def test_case_insensitive(self):
            self.assertIn("Carbamazepine + Erythromycin", check_interactions_text("CARBAMAZEPINE, Erythromycin"))
        def test_requires_two_meds(self):
            self.assertIn("على الأقل", check_interactions_text("warfarin"))
        def test_positive_calculation(self):
            self.assertIn("100.000 mg", dose_calculator(20, 5, None))
        def test_maximum_applied(self):
            self.assertIn("50.000 mg", dose_calculator(20, 5, 50))
        def test_invalid_zero_weight(self):
            self.assertIn("غير صالحة", dose_calculator(0, 5, None))
        def test_all_rules_unverified(self):
            self.assertTrue(all(not r["verified"] for r in INTERACTION_RULES))
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(BasicTests))
    stream = __import__("io").StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)
    return stream.getvalue() + f"\nالنتيجة: {result.testsRun-result.failures-result.errors} ناجح، {len(result.failures)+len(result.errors)} فاشل."

with gr.Blocks(title="Clinical Pharmacy Assistant OS", theme=gr.themes.Soft(),
               css="body {direction: rtl;} .gradio-container {direction: rtl;}") as demo:
    gr.Markdown("""
    # 💊 Clinical Pharmacy Assistant OS
    ### المساعد الإكلينيكي الصيدلاني — نسخة تجريبية لـ Google Colab

    **تنبيه سلامة:** نموذج أولي تعليمي غير معتمد لاتخاذ القرار العلاجي. قاعدة التداخلات محدودة وغير متحققة استقلاليًا.
    لا تدخل بيانات تعريفية حقيقية للمرضى، ولا تغيّر علاجًا أو جرعة بناءً على مخرجات هذا النموذج.
    """)
    with gr.Tab("الرئيسية"):
        gr.Markdown(f"""
        - **قاعدة البيانات:** SQLite في `{DB_PATH}`
        - **قواعد التداخل:** {len(INTERACTION_RULES)} قواعد توضيحية فقط
        - **البيانات:** تبقى في مساحة جلسة Colab ما لم تنقل قاعدة البيانات إلى تخزين دائم
        - **الذكاء الاصطناعي:** غير موصول في هذه النسخة؛ لا يوجد ادعاء باستدعاء Gemini أو قواعد تجارية.
        """)
        refresh = gr.Button("تحديث الجداول")
        patients_table = gr.Dataframe(headers=["ID","رمز المريض","العمر","الوزن kg","الجنس","التشخيصات","عدد الأدوية"], value=list_patients, interactive=False)
        meds_table = gr.Dataframe(headers=["ID","رقم المريض","رمز المريض","الاسم العلمي","التجاري","التركيز","الجرعة المكتوبة","الطريق","التكرار"], value=list_medications, interactive=False)
        refs_table = gr.Dataframe(headers=["ID","المرجع","الجهة","الإصدار","الرابط","حالة الوصول","ملاحظات التحقق","تاريخ التسجيل"], value=list_references, interactive=False)
        refresh.click(fn=list_patients, outputs=patients_table)
        refresh.click(fn=list_medications, outputs=meds_table)
        refresh.click(fn=list_references, outputs=refs_table)

    with gr.Tab("ملف مريض جديد"):
        gr.Markdown("استخدم رمزًا داخليًا غير معرّف بدل الاسم الكامل.")
        p_code = gr.Textbox(label="رمز المريض (مطلوب)")
        with gr.Row():
            p_age = gr.Textbox(label="العمر بالسنوات (اختياري)")
            p_weight = gr.Textbox(label="الوزن بالكيلوغرام (اختياري)")
            p_sex = gr.Dropdown(["غير محدد","ذكر","أنثى","آخر/غير مذكور"], value="غير محدد", label="الجنس")
        p_allergies = gr.Textbox(label="الحساسيات الدوائية", lines=2)
        p_diagnoses = gr.Textbox(label="التشخيصات", lines=2)
        p_notes = gr.Textbox(label="ملاحظات غير تعريفية", lines=2)
        p_submit = gr.Button("حفظ ملف المريض", variant="primary")
        p_msg = gr.Textbox(label="النتيجة", interactive=False)
        p_submit.click(add_patient, [p_code,p_age,p_weight,p_sex,p_allergies,p_diagnoses,p_notes], [p_msg,patients_table])

    with gr.Tab("إضافة دواء"):
        med_pid = gr.Number(label="رقم ملف المريض ID", precision=0)
        with gr.Row():
            med_generic = gr.Textbox(label="الاسم العلمي بالإنجليزية (مطلوب)")
            med_brand = gr.Textbox(label="الاسم التجاري (اختياري)")
        with gr.Row():
            med_strength = gr.Textbox(label="التركيز")
            med_dose = gr.Textbox(label="الجرعة المكتوبة")
            med_route = gr.Textbox(label="طريق الإعطاء")
            med_frequency = gr.Textbox(label="التكرار")
        med_submit = gr.Button("إضافة الدواء", variant="primary")
        med_msg = gr.Textbox(label="النتيجة", interactive=False)
        med_submit.click(add_medication, [med_pid,med_generic,med_brand,med_strength,med_dose,med_route,med_frequency], [med_msg,meds_table])

    with gr.Tab("فحص التداخلات"):
        gr.Markdown("اكتب الأسماء العلمية الإنجليزية مفصولة بفواصل. هذه النسخة تفحص ثلاث قواعد توضيحية فقط.")
        interaction_input = gr.Textbox(label="الأدوية", placeholder="carbamazepine, erythromycin, warfarin", lines=3)
        interaction_button = gr.Button("فحص القواعد المحلية")
        interaction_output = gr.Textbox(label="النتيجة", lines=14, interactive=False)
        interaction_button.click(check_interactions_text, interaction_input, interaction_output)
        gr.Markdown("### فحص أدوية ملف مريض")
        review_pid = gr.Number(label="رقم ملف المريض ID", precision=0)
        review_button = gr.Button("فحص الأدوية المسجلة وحفظ المراجعة")
        review_output = gr.Textbox(label="نتيجة مراجعة الملف", lines=14, interactive=False)
        review_button.click(check_patient_interactions, review_pid, review_output)

    with gr.Tab("حاسبة الوزن"):
        gr.Markdown("حساب رياضي فقط. أدخل جرعة mg/kg/dose مأخوذة مسبقًا من مرجع موثوق، وليس من تخمين النموذج.")
        with gr.Row():
            calc_weight = gr.Number(label="الوزن (kg)", value=10, minimum=0)
            calc_dose = gr.Number(label="الجرعة المدخلة (mg/kg/dose)", value=5, minimum=0)
            calc_max = gr.Textbox(label="الحد الأقصى للجرعة الواحدة mg (اختياري)")
        calc_button = gr.Button("حساب", variant="primary")
        calc_output = gr.Textbox(label="الناتج", lines=5, interactive=False)
        calc_button.click(dose_calculator, [calc_weight,calc_dose,calc_max], calc_output)

    with gr.Tab("سجل المراجع"):
        ref_title = gr.Textbox(label="اسم المرجع (مطلوب)")
        with gr.Row():
            ref_publisher = gr.Textbox(label="الناشر/الجهة")
            ref_edition = gr.Textbox(label="الإصدار أو تاريخ المراجعة")
        ref_url = gr.Textbox(label="رابط المرجع")
        ref_status = gr.Dropdown(["غير محدد","متاح علنًا","اشتراك مرخص","ملف مملوك للمستخدم","يحتاج تحقق"], value="غير محدد", label="حالة الوصول/الترخيص")
        ref_notes = gr.Textbox(label="ملاحظات التحقق والمراجعة", lines=3)
        ref_submit = gr.Button("حفظ المرجع")
        ref_msg = gr.Textbox(label="النتيجة", interactive=False)
        ref_submit.click(add_reference, [ref_title,ref_publisher,ref_edition,ref_url,ref_status,ref_notes], [ref_msg,refs_table])

    with gr.Tab("اختبارات النظام"):
        gr.Markdown("اختبارات برمجية أساسية للوظائف الحالية فقط، وليست تحققًا من السلامة السريرية.")
        test_button = gr.Button("تشغيل الاختبارات", variant="primary")
        test_output = gr.Textbox(label="نتائج الاختبار", lines=18, interactive=False)
        test_button.click(run_tests, outputs=test_output)

    with gr.Tab("نسخ احتياطي"):
        gr.Markdown("يمكن تنزيل قاعدة SQLite الحالية كملف. احفظها في مكان آمن ولا تشاركها إذا احتوت على بيانات حساسة.")
        backup_button = gr.Button("إنشاء نسخة احتياطية")
        backup_file = gr.File(label="ملف قاعدة البيانات")
        def make_backup():
            import shutil
            path = "/content/clinical_pharmacy_backup.db"
            shutil.copy2(DB_PATH, path)
            return path
        backup_button.click(make_backup, outputs=backup_file)

print("تهيئة التطبيق اكتملت. اضغط تشغيل الخلية وانتظر ظهور رابط Gradio.")
demo.launch(share=True, debug=True)
